# Breadth Thrust Indicator：用 qust 衡量市场宽度突增

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


来源参考：[Investopedia](https://www.investopedia.com/terms/b/breadth-thrust-indicator.asp)

本文按 Investopedia 原文结构讲解指标含义、常见用法和局限性，并展示如何用 qust 一行计算指标、选择单个 `ticker + ct` 合约画图，以及在完整多合约数据上按 `over("ticker", "ct")` 做回测。


## 1. Investopedia 原文内容完整改写：Breadth Thrust Indicator

### 什么是 Breadth Thrust
Breadth Thrust 是市场宽度指标，用来衡量上涨参与度是否突然扩张。它不是看某一只股票或某一个合约的价格，而是看市场里有多少标的在上涨。常见计算会用上涨家数除以上涨家数加下跌家数，再对这个比例做短期平滑。

### 指标想表达什么
指数上涨有时可能只由少数权重股推动，这种上涨的广度并不强。Breadth Thrust 关注的是“很多标的同时转强”的情况。如果短时间内上涨家数占比从低位快速升到高位，说明买盘扩散到更广范围，市场内部力量可能发生变化。

### 经典阈值思想
常见 Zweig Breadth Thrust 读法会观察一个短周期宽度均线是否在较短窗口内从 40% 以下上升到 61.5% 以上。低于 40% 说明市场宽度偏弱，高于 61.5% 说明上涨参与度明显扩大。重点不是单日比例，而是短时间内从弱到强的快速切换。

### 计算过程
先得到 advancing 和 declining 数量，然后计算 `advancing / (advancing + declining)`。再对这个比例做 rolling mean。最后检测 rolling mean 是否在指定窗口内曾经低于低阈值，并在当前上穿高阈值。

### 如何使用
它常用于判断大盘或组合层面的风险偏好，而不是单个价格形态。出现 Breadth Thrust 后，交易者可能认为市场从极弱状态进入广泛修复状态。但它通常作为环境过滤器或仓位背景，而不是单独的入场信号。

### 局限性
市场宽度数据的定义很重要。不同交易所、指数成分、过滤规则会得到不同结果。阈值也来自经验，并非所有市场都适用。对期货、行业组合或自定义股票池使用时，要明确 advancing/declining 的来源。

## 2. 从文章到 qust 算子的落地

qust 的 `breadth_thrust` 接收两列：上涨数量和下跌数量。示例里用期货合约横截面的上涨/下跌数量演示方法；真实股票市场应替换成指数成分或交易所的上涨/下跌家数。

## 3. qust 一行调用

```python
col("advancing", "declining").investopedia.breadth_thrust()
```

输入列顺序：`advancing, declining`。

输出列：`breadth_ratio`, `breadth_thrust`, `breadth_thrust_signal`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实GitHub K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
market_breadth = (
    raw
    .with_columns(
        ret=pl.col("close").pct_change().over(["ticker", "ct"]),
    )
    .group_by("datetime")
    .agg(
        (pl.col("ret") > 0).sum().cast(pl.Float64).alias("advancing"),
        (pl.col("ret") < 0).sum().cast(pl.Float64).alias("declining"),
    )
    .with_columns((pl.col("advancing") + pl.col("declining")).alias("active_count"))
    .filter(pl.col("active_count") > 0)
    .drop("active_count")
    .sort("datetime")
)

indicator_expr = col("advancing", "declining").investopedia.breadth_thrust(
    ma_period=10,
    signal_window=10,
)
breadth_data = col.with_cols(indicator_expr).calc_data(market_breadth)
plot_data = breadth_data.tail(2500)

summary = col(
    col("breadth_thrust_signal").cast(pl.UInt32).sum().alias("breadth_thrust_signal_count"),
    col("breadth_ratio").mean().alias("avg_breadth_ratio"),
    col("breadth_thrust").max().alias("max_breadth_thrust"),
).calc_data(breadth_data)

summary

breadth_thrust_signal_count,avg_breadth_ratio,max_breadth_thrust
u32,f64,f64
17817,0.499647,1.0


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 Jupyter 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
breadth_plot = col(
    col("datetime", "breadth_ratio", "breadth_thrust")
        .monitor("breadth", show_axis_label=True)
        .line(),
    col("datetime", "breadth_thrust", "breadth_thrust_signal")
        .monitor("breadth", show_axis_label=True)
        .mark(shape=mark_shape.triangle_up, color="#50fa7b", width=0.4),
).monitor.make_monitor("black").monitor.add_grid([
    ["breadth"],
]).runtime()

breadth_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Breadth Thrust 策略回测

Breadth Thrust 是市场宽度信号，不来自单个合约。先用 Polars 做市场宽度数据准备，再用 qust `breadth_thrust` 算子生成宽度信号，最后把信号按 `datetime` join 回每个合约。当前样本里宽度突增后做多为正：信号后移一根 K 线入场，2% 止盈、1.5% 止损，并用 `col("hold") / col.all.fp.vol_pms()` 做持仓归一化。

In [4]:
TAKE_PROFIT = 0.02
STOP_LOSS = 0.015

market_breadth_for_strategy = (
    raw
    .with_columns(ret=pl.col("close").pct_change().over(["ticker", "ct"]))
    .group_by("datetime")
    .agg(
        (pl.col("ret") > 0).sum().cast(pl.Float64).alias("advancing"),
        (pl.col("ret") < 0).sum().cast(pl.Float64).alias("declining"),
    )
    .with_columns((pl.col("advancing") + pl.col("declining")).alias("active_count"))
    .filter(pl.col("active_count") > 0)
    .drop("active_count")
    .sort("datetime")
)

breadth_signal = (
    col
    .with_cols(col("advancing", "declining").investopedia.breadth_thrust(ma_period=10, signal_window=10))
    .with_cols(
        col("breadth_thrust_signal").fill_null(col.lit(False)).alias("breadth_open_long"),
    )
    .select("datetime", "breadth_open_long", "breadth_ratio", "breadth_thrust")
    .calc_data(market_breadth_for_strategy)
)

strategy_input = (
    raw
    .join(breadth_signal, on="datetime", how="left")
    .with_columns(pl.col("breadth_open_long").fill_null(False))
)

strategy_daily_expr = (
    col
    .with_cols(
        col("breadth_open_long").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
        col.lit(False).alias("open_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
        col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
        col.lit(False).alias("take_profit_short"),
        col.lit(False).alias("stop_loss_short"),
    )
    .with_cols(
        (col("take_profit_long") | col("stop_loss_long")).fill_null(col.lit(False)).alias("exit_long_sig"),
        col.lit(False).alias("exit_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra.to_hold_two_sides()
            .expanding()
            .alias("hold")
    )
    .with_cols((col("hold") / col.all.fp.vol_pms()).alias("hold"))
    .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
    .over("ticker", "ct")
    .select(
        col("pnl")
            .sum()
            .group_by(col("datetime").dt.date().alias("date"))
            .batch.sort("date")
            .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
            .select("date", "pnl", "pnl_cum")
    )
)

strategy_daily = strategy_daily_expr.calc_data(strategy_input)
strategy_stats = col("date", "pnl").bt.returns_stats(periods_per_year=252).calc_data(strategy_daily)

print("strategy_input shape:", strategy_input.shape)
print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_input shape: (408782, 11)
strategy_daily shape: (859, 3)


metric,value,value_float
str,str,f64
"""Start Index""","""2022-01-04""",null
"""End Index""","""2024-12-31""",null
"""Total Duration""","""1092 days, 0:00:00""",null
"""Total Return [%]""","""-5.706035535512229e+148""",-5.7060e148
"""Benchmark Return [%]""",null,null
"""Annualized Return [%]""",null,null
"""Annualized Volatility [%]""","""3987.3801682096014""",3987.380168
"""Max Drawdown [%]""","""54019.29728207572""",54019.297282
…,…,…


In [5]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,-0.279737,70.055148
2024-12-19,-1.5994,68.455747
2024-12-20,-0.789056,67.666692
2024-12-21,-0.125345,67.541346
2024-12-23,-0.003989,67.537357
2024-12-24,1.935381,69.472738
2024-12-25,-1.474957,67.997781
2024-12-26,-0.15077,67.847012
2024-12-27,-2.337743,65.509269


## 7. 策略 PnL 曲线

下面用 qust monitor 画策略累计 PnL 和每日 PnL。Breadth Thrust 是全市场信号，所以每个合约会在同一市场宽度事件后进入自己的持仓管理流程。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。